Post this test for issues in data curation for training: 

Split between training, test and validation pairs and documents 

Triplet formation from pairs 

Random Negative sampling 

Negative Mining 

Manual addition to validation set 

Check if the document is not truncated post tokenization using bert-base using max_doc_len = 512 : check krwaya tha hona to nahi chahiye

### from the original training and validation set (before oversamplimng): 
#### current procedure: 
1. for each (query,label) sample each set in defined ratios 
2. pos neg ratio for each set is required 
3. for each split triplet generator is called 
4. inside triplet generator a index mapping is created 
5. all the possible rule based negatives are crated each time tripletgenerator is initialized 
6. triplets are generated for each split 
7. per query, positive (<max_positives) : negatives are sampled 
8. ngatives are sampled in order or dictionary strategy passd : hence if there aren't enough negatives to be sampled to divide into int groups then initial ones will be sampled
#### verify: 
1. All splits have hard negatives 
2. All 

In [ ]:
from train_colbert import *
experiment_dir='../data/sampling/'
data_dir = experiment_dir + "data"
os.makedirs(data_dir, exist_ok=True)
s=time.time()

labeled_pairs_path='/home/ec2-user/SageMaker/data/labelled_pairs.all.pkl'
collections_path='/home/ec2-user/SageMaker/data/collections.all.tsv'

labeled_pairs, documents=load_real_data(labeled_pairs_path,collections_path,aspect_delimiter='||')
print(f'time for loading = {time.time()-s}')
s=time.time()
# 3. Split the dataset
print("Splitting dataset...")
negative_miner=SimpleMiner(language_code='other')

splitter = TripletDatasetSplitter(
    labeled_pairs=labeled_pairs,
    collection=documents,
    negative_miner=negative_miner,  # No need for a miner in this example
    aspect_delimiter="||",
    train_val_test_ratio=(0.8, 0.1, 0.1),
    train_pos_neg_ratio=64,
    val_pos_neg_ratio=16,
    test_pos_neg_ratio=8,
    seed=42,
    debug=True
)

print(f'time for splitting = {time.time()-s}')
# Split and process the dataset
s=time.time()

datasets = splitter.process_data(
    output_dir=data_dir,
    max_triplets_per_query=10000,  # Increased from 10
    train_max_positives=100,  # Increased from 2
    test_max_positives=20,
    val_max_positives=10
)

In [2]:
16*0.25

4.0

In [ ]:
def _sample_miner_negatives(self, query, positive, count, already_sampled=None):
        """Sample negatives using the provided negative miner."""
        if already_sampled is None:
            already_sampled = set()
        
        if not self.negative_miner:
            return []
            
        # Get hard negatives from miner
        hard_negatives = self.negative_miner.mine_hard_negatives(query)
        
        # Filter out positives, already sampled, and documents not in our collection
        candidates = [
            neg for neg in hard_negatives 
            if (neg in self.passage_map and
                neg not in self.query_positives.get(query, set()) and 
                neg != positive and 
                neg not in already_sampled)
        ]
        
        return random.sample(candidates, min(count, len(candidates))) if candidates else []

In [1]:
k=[1,2,3]
k[1:5]

[2, 3]